In [ ]:
import scanpy as sc

# 1. Load your processed file
file_path = "perturb_processed.h5ad"
# file_path = "./data/project_dataset/perturb_processed.h5ad"
adata = sc.read_h5ad(file_path)
adata

In [ ]:
# 2. Check if 'ctrl' exists (Crucial for DE analysis)
if 'ctrl' not in adata.obs['condition'].values:
    raise ValueError("Error: No 'ctrl' cells found in adata.obs['condition']. \
    Please run the previous fix script to map 'pre-IR' (or similar) to 'ctrl'.")

Calculating Differential Expression stats (required by PertData loader)

In [ ]:
# 3. Create a temporary normalized copy for DE calculation
# (We use a copy so we don't accidentally overwrite your raw counts in adata.X)
adata_norm = adata.copy()
sc.pp.normalize_total(adata_norm, target_sum=1e4)
sc.pp.log1p(adata_norm)

# 4. Run Rank Genes Groups (Compare everything vs 'ctrl')
sc.tl.rank_genes_groups(
    adata_norm,
    groupby='condition',
    reference='ctrl',
    method='wilcoxon',
    use_raw=False
)

# 5. Inject the result into the main adata object
# The loader specifically looks for 'rank_genes_groups_cov_all'
adata.uns['rank_genes_groups_cov_all'] = adata_norm.uns['rank_genes_groups']

In [ ]:
adata

In [ ]:
# save_path = "./data/project_dataset/perturb_processed.h5ad"
save_path = "perturb_processed.h5ad"
adata.write_h5ad(save_path)

In [ ]:
adata